## Question 9. Transformer Decoder Block and Complete Architecture

A post-normalization decoder block has three sublayers. Causal self-attention stops each target position from reading future target positions. Cross-attention then reads the encoder output. A position-wise ReLU feed-forward network follows. Each sublayer is added back to its input and layer-normalized. The output length matches the decoder input.

## Code Execution

In [1]:
import numpy as np

**Random source.** Seed 0 makes the parameter draw and both input sequences repeatable.

In [2]:
RNG = np.random.default_rng(0)

**Attention from Question 6.** Masked self-attention sets `causal=True`. Cross-attention leaves the mask off, so every decoder position may read every encoder position.

In [3]:
def stable_softmax(x, axis=-1):
    """Numerically stable softmax along one axis."""
    x = np.asarray(x, dtype=float)
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_shifted = np.exp(shifted)
    return exp_shifted / np.sum(exp_shifted, axis=axis, keepdims=True)


def attention_with_mask(Q, K, V, causal=False):
    """One head: Q (nq, dk), K (nk, dk), V (nk, dv)."""
    scores = (Q @ K.T) / np.sqrt(Q.shape[1])
    if causal:
        if Q.shape[0] != K.shape[0]:
            raise ValueError("Causal mask requires equal query/key lengths")
        future = np.triu(np.ones(scores.shape, dtype=bool), k=1)
        scores = np.where(future, -np.inf, scores)
    weights = stable_softmax(scores, axis=-1)
    return weights @ V, weights


def multi_head_attention_general(
    query_states, key_value_states, W_Q, W_K, W_V, W_O, num_heads, causal=False
):
    """Packed-head attention: output (nq, d_model), weights (H, nq, nk)."""
    query_states = np.asarray(query_states, dtype=float)
    key_value_states = np.asarray(key_value_states, dtype=float)
    if (
        query_states.ndim != 2
        or key_value_states.ndim != 2
        or query_states.shape[1] != key_value_states.shape[1]
    ):
        raise ValueError("Input states must be 2D with the same model width")
    d_model = query_states.shape[1]
    if num_heads < 1 or d_model % num_heads:
        raise ValueError("num_heads must divide d_model")
    for W in (W_Q, W_K, W_V, W_O):
        if np.shape(W) != (d_model, d_model):
            raise ValueError("Packed projection matrices must be (d_model, d_model)")
    Q = query_states @ W_Q
    K = key_value_states @ W_K
    V = key_value_states @ W_V
    d_head = d_model // num_heads
    outputs, weights = [], []
    for h in range(num_heads):
        sl = slice(h * d_head, (h + 1) * d_head)
        head_output, head_weights = attention_with_mask(
            Q[:, sl], K[:, sl], V[:, sl], causal=causal
        )
        outputs.append(head_output)
        weights.append(head_weights)
    concatenated = np.concatenate(outputs, axis=-1)
    return concatenated @ W_O, np.stack(weights, axis=0)

**Layer norm.** Same token-wise normalization as Question 8. $\gamma$ and $\beta$ rescale and shift each token after centering it.

In [4]:
def layer_norm(X, gamma, beta, eps=1e-5):
    """Normalize each token over its final feature axis."""
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=-1, keepdims=True)
    var = ((X - mean) ** 2).mean(axis=-1, keepdims=True)
    return (X - mean) / np.sqrt(var + eps) * gamma + beta

**Feed-forward.** The same two linear maps and a ReLU are applied to every decoder token.

In [5]:
def feed_forward(X, W1, b1, W2, b2):
    """(n, d) -> (n, d_ff) -> (n, d), same weights for every token."""
    hidden = np.maximum(0, X @ W1 + b1)
    return hidden @ W2 + b2

**Decoder block.** Causal self-attention uses the decoder states as queries and keys. Cross-attention uses the updated decoder states as queries and the encoder states as keys and values. The feed-forward network is the third sublayer. Each step adds its output back and applies layer norm.

In [7]:
def transformer_decoder_block(decoder_input, encoder_output, params):
    """Post-normalization decoder: (n, d) + encoder (m, d) -> (n, d)."""
    H0, _ = multi_head_attention_general(
        decoder_input,
        decoder_input,
        params["self_W_Q"],
        params["self_W_K"],
        params["self_W_V"],
        params["self_W_O"],
        params["num_heads"],
        causal=True,
    )
    H1 = layer_norm(decoder_input + H0, params["gamma1"], params["beta1"])
    Hcross, _ = multi_head_attention_general(
        H1,
        encoder_output,
        params["cross_W_Q"],
        params["cross_W_K"],
        params["cross_W_V"],
        params["cross_W_O"],
        params["num_heads"],
    )
    H2 = layer_norm(H1 + Hcross, params["gamma2"], params["beta2"])
    Hffn = feed_forward(H2, params["W1"], params["b1"], params["W2"], params["b2"])
    return layer_norm(H2 + Hffn, params["gamma3"], params["beta3"])

**Random parameters.** Separate projections are drawn for self-attention and cross-attention. The feed-forward scales match Question 8. Three layer-norm pairs start at scale one and shift zero.

In [8]:
def random_decoder_params(d_model=8, d_ff=16, num_heads=2):
    scale = 1 / np.sqrt(d_model)
    names = (
        "self_W_Q",
        "self_W_K",
        "self_W_V",
        "self_W_O",
        "cross_W_Q",
        "cross_W_K",
        "cross_W_V",
        "cross_W_O",
    )
    params = {
        name: RNG.normal(0, scale, (d_model, d_model)) for name in names
    }
    params.update(
        W1=RNG.normal(0, scale, (d_model, d_ff)),
        b1=np.zeros(d_ff),
        W2=RNG.normal(0, 1 / np.sqrt(d_ff), (d_ff, d_model)),
        b2=np.zeros(d_model),
        num_heads=num_heads,
    )
    for k in (1, 2, 3):
        params[f"gamma{k}"] = np.ones(d_model)
        params[f"beta{k}"] = np.zeros(d_model)
    return params

**Inputs.** Three decoder tokens and five encoder tokens, each of width $8$.

In [9]:
decoder_input = RNG.normal(size=(3, 8))
encoder_output = RNG.normal(size=(5, 8))
p9 = random_decoder_params(d_model=8, num_heads=2)

**Block output.** The decoder length stays $3$. The encoder length does not appear in the output shape.

In [10]:
Y9 = transformer_decoder_block(decoder_input, encoder_output, p9)
print("Q9 decoder, encoder, output:", decoder_input.shape, encoder_output.shape, Y9.shape)

Q9 decoder, encoder, output: (3, 8) (5, 8) (3, 8)


**Weight shapes.** Masked self-attention on the decoder input is $2\times 3\times 3$. Cross-attention from that same input to the encoder states is $2\times 3\times 5$.

In [11]:
_, masked9 = multi_head_attention_general(
    decoder_input,
    decoder_input,
    p9["self_W_Q"],
    p9["self_W_K"],
    p9["self_W_V"],
    p9["self_W_O"],
    2,
    causal=True,
)
_, cross9 = multi_head_attention_general(
    decoder_input,
    encoder_output,
    p9["cross_W_Q"],
    p9["cross_W_K"],
    p9["cross_W_V"],
    p9["cross_W_O"],
    2,
)
print("Q9 masked/cross weight shapes:", masked9.shape, cross9.shape)

Q9 masked/cross weight shapes: (2, 3, 3) (2, 3, 5)


**Check.** The output is $(3, 8)$. Positions strictly above the diagonal of the masked weights are $0$.

In [12]:
future_rows, future_cols = np.triu_indices(3, k=1)
assert Y9.shape == (3, 8) and masked9.shape == (2, 3, 3) and cross9.shape == (2, 3, 5)
assert np.all(masked9[:, future_rows, future_cols] == 0)
assert np.all(np.isfinite(Y9))
print("Decoder block check passed.")

Decoder block check passed.
